In [59]:
import numpy as np

class NearestNeighbor:
    """
    A simple 1-Nearest Neighbor classifier implementation from scratch.
    
    This classifier uses the L1 (Manhattan) distance metric to find the closest
    training sample to each test sample and assigns the same label.
    
    Attributes:
        Xtr (np.ndarray): Training feature matrix (N x D)
        ytr (np.ndarray): Training labels vector (N,)
    """
    
    def __init__(self):
        """Initialize the classifier with empty training data."""
        self.Xtr = None
        self.ytr = None
        
    def train(self, X, y):
        """
        Store the training data (lazy learning - no actual training occurs).
        
        Args:
            X (np.ndarray): Training features with shape (N, D)
                          N = number of training samples
                          D = number of features per sample
            y (np.ndarray): Training labels with shape (N,)
                          Contains class labels for each training sample
        """
        # Simply store the training data - no computation needed
        # This is characteristic of "lazy learning" algorithms
        self.Xtr = X  # Feature matrix: rows = samples, columns = features
        self.ytr = y  # Label vector: one label per sample

    def predict(self, X):
        """
        Predict class labels for test samples using 1-nearest neighbor.
        
        For each test sample, finds the training sample with minimum L1 distance
        and assigns its label to the test sample.
        
        Args:
            X (np.ndarray): Test features with shape (M, D)
                          M = number of test samples
                          D = number of features (must match training data)
        
        Returns:
            np.ndarray: Predicted labels with shape (M,)
                       Same dtype as training labels
        """
        num_test = X.shape[0]  # Number of test samples to classify
        
        # Initialize prediction array with same type as training labels
        Ypred = np.zeros(num_test, dtype=self.ytr.dtype)
        
        # Process each test sample individually
        for i in range(num_test):
            # Calculate L1 (Manhattan) distance from test sample to all training samples
            # 
            # Step-by-step breakdown:
            # 1. self.Xtr - X[i,:] : subtract test sample from each training sample
            #    Shape: (N_train, D) - (D,) = (N_train, D) via broadcasting
            # 2. np.abs(...) : take absolute value element-wise
            # 3. np.sum(..., axis=1) : sum across features for each training sample
            #    Result: (N_train,) array of distances
            
            # L1 distance formula: Σ|x_train_i - x_test_i| for each feature i
            distances = np.sum(np.abs(self.Xtr - X[i,:]), axis=1)
            
            # Find the index of the training sample with minimum distance
            min_index = np.argmin(distances)
            
            # Assign the label of the nearest neighbor to this test sample
            Ypred[i] = self.ytr[min_index]
            
        return Ypred

In [60]:
#import numpy as np

# Generate a small N × D matrix X and corresponding y vector
np.random.seed(42)  # For reproducible results

# Parameters
N = 10  # Number of samples
D = 3   # Number of features

# Generate random feature matrix X (N × D)
X = np.random.randn(N, D)

# Generate random labels y (binary classification: 0 or 1)
y = np.random.randint(0, 2, N)

print("Feature matrix X (shape:", X.shape, "):")
print(X)
print("\nLabel vector y (shape:", y.shape, "):")
print(y)

# Let's also create some test data
X_test = np.random.randn(4, D)  # 3 test samples with same number of features
print("\nTest data X_test (shape:", X_test.shape, "):")
print(X_test)

Feature matrix X (shape: (10, 3) ):
[[ 0.49671415 -0.1382643   0.64768854]
 [ 1.52302986 -0.23415337 -0.23413696]
 [ 1.57921282  0.76743473 -0.46947439]
 [ 0.54256004 -0.46341769 -0.46572975]
 [ 0.24196227 -1.91328024 -1.72491783]
 [-0.56228753 -1.01283112  0.31424733]
 [-0.90802408 -1.4123037   1.46564877]
 [-0.2257763   0.0675282  -1.42474819]
 [-0.54438272  0.11092259 -1.15099358]
 [ 0.37569802 -0.60063869 -0.29169375]]

Label vector y (shape: (10,) ):
[1 1 0 1 0 1 0 1 0 0]

Test data X_test (shape: (4, 3) ):
[[-0.22945045  0.38934891 -1.26511911]
 [ 1.09199226  2.77831304  1.19363972]
 [ 0.21863832  0.88176104 -1.00908534]
 [-1.58329421  0.77370042 -0.53814166]]


In [61]:
# Let's break down the distance calculation step by step
print("Understanding: distances = np.sum(np.square(self.Xtr - X[i,:]), axis=1)")
print("=" * 60)

# Let's use our actual data to demonstrate
print("Training data shape (Xtr):", X.shape)  # (10, 3)
print("Single test sample shape (X[i,:]):", X_test[0,:].shape)  # (3,)
print()

# Step 1: Broadcasting subtraction
test_sample = X_test[0,:]  # First test sample
print("Step 1: self.Xtr - X[i,:]")
print("This subtracts the test sample from EACH training sample")
differences = X - test_sample
print("Shape after subtraction:", differences.shape)  # (10, 3)
print("First few differences:\n", differences[:3])
print()

# Step 2: Square the differences
print("Step 2: np.square(...)")
print("Square each difference element-wise")
squared_diffs = np.square(differences)
print("Shape after squaring:", squared_diffs.shape)  # (10, 3)
print("First few squared differences:\n", squared_diffs[:3])
print()

# Step 3: Sum along axis=1 (across features)
print("Step 3: np.sum(..., axis=1)")
print("Sum across features (axis=1) for each training sample")
distances = np.sum(squared_diffs, axis=1)
print("Shape after summing:", distances.shape)  # (10,)
print("Final distances:", distances)
print()

print("This gives us the SQUARED Euclidean distance from the test sample")
print("to each of the", len(distances), "training samples")
print("The smallest distance indicates the nearest neighbor!")

Understanding: distances = np.sum(np.square(self.Xtr - X[i,:]), axis=1)
Training data shape (Xtr): (10, 3)
Single test sample shape (X[i,:]): (3,)

Step 1: self.Xtr - X[i,:]
This subtracts the test sample from EACH training sample
Shape after subtraction: (10, 3)
First few differences:
 [[ 0.72616461 -0.52761321  1.91280765]
 [ 1.75248031 -0.62350229  1.03098216]
 [ 1.80866327  0.37808582  0.79564473]]

Step 2: np.square(...)
Square each difference element-wise
Shape after squaring: (10, 3)
First few squared differences:
 [[0.52731504 0.2783757  3.65883311]
 [3.07118724 0.3887551  1.06292421]
 [3.27126282 0.14294888 0.63305053]]

Step 3: np.sum(..., axis=1)
Sum across features (axis=1) for each training sample
Shape after summing: (10,)
Final distances: [ 4.46452385  4.52286655  4.04726224  1.96223444  5.73574586  4.57128774
 11.16350753  0.12906351  0.18972819  2.29383707]

This gives us the SQUARED Euclidean distance from the test sample
to each of the 10 training samples
The smalles

In [62]:
X_test.shape[0]

4

In [63]:
# Create and train the NearestNeighbor classifier
nn = NearestNeighbor()

# Train the classifier with our generated data
print("Training the classifier...")
nn.train(X, y)
print("Training complete!")

# Make predictions on test data
print("\nMaking predictions on test data...")
predictions = nn.predict(X_test)

print("Test data shape:", X_test.shape)
print("Predictions:", predictions)
print("Predictions shape:", predictions.shape)

Training the classifier...
Training complete!

Making predictions on test data...
Test data shape: (4, 3)
Predictions: [1 1 1 0]
Predictions shape: (4,)


In [64]:
# Create a more realistic dataset: Iris-like data
print("Creating a realistic Iris-like dataset...")
print("=" * 50)

np.random.seed(123)  # Different seed for variety

# Create 3 classes of flowers with different characteristics
n_samples_per_class = 50
n_features = 4  # sepal length, sepal width, petal length, petal width

# Class 0: Small flowers (like Setosa)
class0_data = np.random.normal([5.0, 3.5, 1.5, 0.3], [0.3, 0.3, 0.2, 0.1], (n_samples_per_class, n_features))
class0_labels = np.zeros(n_samples_per_class, dtype=int)

# Class 1: Medium flowers (like Versicolor) 
class1_data = np.random.normal([6.0, 2.8, 4.0, 1.3], [0.4, 0.3, 0.3, 0.2], (n_samples_per_class, n_features))
class1_labels = np.ones(n_samples_per_class, dtype=int)

# Class 2: Large flowers (like Virginica)
class2_data = np.random.normal([6.5, 3.0, 5.5, 2.0], [0.4, 0.3, 0.4, 0.3], (n_samples_per_class, n_features))
class2_labels = np.full(n_samples_per_class, 2, dtype=int)

# Combine all classes
X_iris = np.vstack([class0_data, class1_data, class2_data])
y_iris = np.hstack([class0_labels, class1_labels, class2_labels])

print(f"Dataset shape: {X_iris.shape}")
print(f"Labels shape: {y_iris.shape}")
print(f"Classes: {np.unique(y_iris)}")
print(f"Samples per class: {np.bincount(y_iris)}")

# Display some statistics
print("\nDataset statistics:")
print("Feature means by class:")
for class_id in range(3):
    class_mask = y_iris == class_id
    class_mean = np.mean(X_iris[class_mask], axis=0)
    print(f"  Class {class_id}: [{class_mean[0]:.2f}, {class_mean[1]:.2f}, {class_mean[2]:.2f}, {class_mean[3]:.2f}]")

print("\nFirst 5 samples from each class:")
for class_id in range(3):
    class_mask = y_iris == class_id
    class_data = X_iris[class_mask]
    print(f"Class {class_id} samples:")
    print(class_data[:5])
    print()

Creating a realistic Iris-like dataset...
Dataset shape: (150, 4)
Labels shape: (150,)
Classes: [0 1 2]
Samples per class: [50 50 50]

Dataset statistics:
Feature means by class:
  Class 0: [5.02, 3.47, 1.54, 0.28]
  Class 1: [5.98, 2.76, 3.92, 1.30]
  Class 2: [6.58, 3.06, 5.46, 1.99]

First 5 samples from each class:
Class 0 samples:
[[4.67431082 3.79920363 1.5565957  0.14937053]
 [4.82641992 3.99543096 1.01466415 0.25710874]
 [5.37978088 3.23997788 1.36422277 0.2905291 ]
 [5.44741689 3.3083294  1.41120361 0.25656487]
 [5.66177902 4.15603583 1.70081078 0.33861864]]

Class 1 samples:
[[6.28132405 2.6205684  4.66021063 1.43765939]
 [5.9974771  2.73800131 3.97404331 1.11693859]
 [5.96191898 2.88360506 4.17386248 1.41593796]
 [5.89004898 2.37517532 3.79926921 1.62243861]
 [6.35842333 2.91088588 3.77161173 1.30072903]]

Class 2 samples:
[[7.11363612 2.84102577 5.30361109 1.60725041]
 [6.49653581 3.29304389 4.79957186 1.80024291]
 [6.5143762  3.25503087 5.6531481  2.09763909]
 [6.41027489 

In [65]:
# Test the NearestNeighbor classifier on the realistic dataset
print("Testing NearestNeighbor on Iris-like dataset...")
print("=" * 50)

# Split the data into train and test sets (80-20 split)
n_total = len(X_iris)
n_train = int(0.8 * n_total)

# Shuffle the data before splitting
shuffle_indices = np.random.permutation(n_total)
X_iris_shuffled = X_iris[shuffle_indices]
y_iris_shuffled = y_iris[shuffle_indices]

# Split into train and test
X_train_iris = X_iris_shuffled[:n_train]
y_train_iris = y_iris_shuffled[:n_train]
X_test_iris = X_iris_shuffled[n_train:]
y_test_iris = y_iris_shuffled[n_train:]

print(f"Training set: {X_train_iris.shape[0]} samples")
print(f"Test set: {X_test_iris.shape[0]} samples")

# Create and train the classifier
nn_iris = NearestNeighbor()
nn_iris.train(X_train_iris, y_train_iris)

# Make predictions
predictions_iris = nn_iris.predict(X_test_iris)

# Calculate accuracy
accuracy = np.mean(predictions_iris == y_test_iris)
print(f"\nAccuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

# Show detailed results
print("\nDetailed results:")
print("True labels:     ", y_test_iris)
print("Predicted labels:", predictions_iris)
print("Correct:         ", predictions_iris == y_test_iris)

# Count correct predictions per class
print("\nPer-class results:")
for class_id in range(3):
    mask = y_test_iris == class_id
    if np.sum(mask) > 0:
        class_accuracy = np.mean(predictions_iris[mask] == y_test_iris[mask])
        n_samples = np.sum(mask)
        print(f"Class {class_id}: {class_accuracy:.3f} accuracy ({np.sum(predictions_iris[mask] == y_test_iris[mask])}/{n_samples} correct)")

Testing NearestNeighbor on Iris-like dataset...
Training set: 120 samples
Test set: 30 samples

Accuracy: 1.000 (100.0%)

Detailed results:
True labels:      [2 0 0 2 0 2 0 0 1 0 0 1 0 1 0 1 0 1 0 0 2 0 2 2 2 0 2 1 2 0]
Predicted labels: [2 0 0 2 0 2 0 0 1 0 0 1 0 1 0 1 0 1 0 0 2 0 2 2 2 0 2 1 2 0]
Correct:          [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True]

Per-class results:
Class 0: 1.000 accuracy (15/15 correct)
Class 1: 1.000 accuracy (6/6 correct)
Class 2: 1.000 accuracy (9/9 correct)


In [66]:
# Load the actual MNIST dataset from the repository
print("Loading MNIST dataset from repository...")
print("=" * 60)

# Load the MNIST data files
mnist_path = "NN_MNIST/NN_MNIST/MNIST/"
X_train_mnist = np.load(mnist_path + "train_data.npy")
y_train_mnist = np.load(mnist_path + "train_labels.npy")
X_test_mnist = np.load(mnist_path + "test_data.npy")
y_test_mnist = np.load(mnist_path + "test_labels.npy")

print(f"Training data shape: {X_train_mnist.shape}")
print(f"Training labels shape: {y_train_mnist.shape}")
print(f"Test data shape: {X_test_mnist.shape}")
print(f"Test labels shape: {y_test_mnist.shape}")
print(f"\nClasses: {np.unique(y_train_mnist)}")
print(f"Training samples per class: {np.bincount(y_train_mnist)}")

# If the data is in image format (N, H, W), flatten it to (N, H*W)
if len(X_train_mnist.shape) == 3:
    print(f"\nFlattening images from shape {X_train_mnist.shape} to 2D...")
    X_train_mnist = X_train_mnist.reshape(X_train_mnist.shape[0], -1)
    X_test_mnist = X_test_mnist.reshape(X_test_mnist.shape[0], -1)
    print(f"New training shape: {X_train_mnist.shape}")
    print(f"New test shape: {X_test_mnist.shape}")

print(f"\nPixel value range: [{X_train_mnist.min():.2f}, {X_train_mnist.max():.2f}]")

Loading MNIST dataset from repository...
Training data shape: (7500, 784)
Training labels shape: (7500,)
Test data shape: (1000, 784)
Test labels shape: (1000,)

Classes: [0 1 2 3 4 5 6 7 8 9]
Training samples per class: [750 750 750 750 750 750 750 750 750 750]

Pixel value range: [0.00, 255.00]


In [67]:
import time

# Test NearestNeighbor on MNIST dataset (with a small subset for speed)
print("Testing NearestNeighbor on MNIST dataset...")
print("=" * 60)

# Use a smaller subset for faster computation
# Full MNIST training is very slow with 1-NN (60,000 training samples)
n_train_subset = 1000  # Use 1000 training samples
n_test_subset = 200    # Use 200 test samples

print(f"Using subset: {n_train_subset} training samples, {n_test_subset} test samples")
print("(Full dataset would take a very long time with 1-NN!)")

X_train_subset = X_train_mnist[:n_train_subset]
y_train_subset = y_train_mnist[:n_train_subset]
X_test_subset = X_test_mnist[:n_test_subset]
y_test_subset = y_test_mnist[:n_test_subset]

print("\nSubset shapes:")
print(f"  Training: {X_train_subset.shape}")
print(f"  Test: {X_test_subset.shape}")

# Train the classifier
print("\nTraining 1-Nearest Neighbor classifier...")
nn_mnist = NearestNeighbor()
nn_mnist.train(X_train_subset, y_train_subset)
print("Training complete (data stored)!")

# Make predictions
print("\nMaking predictions on test set...")
print("This will take a moment...")
#import time
start_time = time.time()
predictions_mnist = nn_mnist.predict(X_test_subset)
elapsed_time = time.time() - start_time

# Calculate accuracy
accuracy = np.mean(predictions_mnist == y_test_subset)
print("\nResults:")
print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Correct: {np.sum(predictions_mnist == y_test_subset)}/{len(y_test_subset)}")
print(f"  Time: {elapsed_time:.2f} seconds")
print(f"  Average per sample: {elapsed_time/len(y_test_subset)*1000:.1f} ms")

# Show confusion for first few predictions
print("\nFirst 10 predictions:")
for i in range(min(10, len(y_test_subset))):
    status = "✓" if predictions_mnist[i] == y_test_subset[i] else "✗"
    print(f"  {status} Sample {i}: predicted {predictions_mnist[i]}, actual {y_test_subset[i]}")

# Per-class accuracy
print("\nPer-digit accuracy:")
for digit in range(10):
    mask = y_test_subset == digit
    if np.sum(mask) > 0:
        digit_accuracy = np.mean(predictions_mnist[mask] == y_test_subset[mask])
        n_correct = np.sum(predictions_mnist[mask] == y_test_subset[mask])
        n_total = np.sum(mask)
        print(f"  Digit {digit}: {digit_accuracy:.3f} ({n_correct}/{n_total})")

Testing NearestNeighbor on MNIST dataset...
Using subset: 1000 training samples, 200 test samples
(Full dataset would take a very long time with 1-NN!)

Subset shapes:
  Training: (1000, 784)
  Test: (200, 784)

Training 1-Nearest Neighbor classifier...
Training complete (data stored)!

Making predictions on test set...
This will take a moment...

Results:
  Accuracy: 0.9100 (91.00%)
  Correct: 182/200
  Time: 0.20 seconds
  Average per sample: 1.0 ms

First 10 predictions:
  ✓ Sample 0: predicted 0, actual 0
  ✓ Sample 1: predicted 2, actual 2
  ✓ Sample 2: predicted 6, actual 6
  ✗ Sample 3: predicted 3, actual 5
  ✓ Sample 4: predicted 9, actual 9
  ✓ Sample 5: predicted 6, actual 6
  ✓ Sample 6: predicted 0, actual 0
  ✓ Sample 7: predicted 7, actual 7
  ✓ Sample 8: predicted 1, actual 1
  ✓ Sample 9: predicted 1, actual 1

Per-digit accuracy:
  Digit 0: 1.000 (22/22)
  Digit 1: 0.952 (20/21)
  Digit 2: 0.864 (19/22)
  Digit 3: 1.000 (17/17)
  Digit 4: 0.917 (22/24)
  Digit 5: 0.88